# 02 — Reading Q, K, V and O: find the copying circuit from the weights alone

**References · Weight-Space Programming** · ~20 min · CPU only · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zenocode-org/ai-masterclass/blob/main/en/references/01_weight_space_programming/02_reading_qk_ov_circuits.ipynb)

In Experiment 3, Astra gets only the **middle blocks** of a small character-level transformer. It is not allowed to run it, yet it names the heads that **copy** earlier text. It does so by multiplying weight matrices together, a method from Elhage et al. (2021) that nobody showed it.

This notebook teaches that method on a model small enough to check everything. It also answers a question from chapter 2: **what do the Query, Key and Value matrices actually learn?**

**By the end you will be able to:**
1. Explain why `W_Q` and `W_K` only matter through their product `W_QK = W_Qᵀ W_K`, and `W_V`, `W_O` through `W_OV = W_O W_V`.
2. Read a head's **OV circuit** (what it moves) and its **QK circuit** (where it looks) as tables over tokens and positions.
3. Find how two heads in different layers **compose** into an algorithm, from weights only.
4. Confirm the reading with attention patterns and ablations (the checks Astra was *not* allowed to run).

**Plan:**
1. The task: copy a repeated segment
2. A transformer made only of attention heads
3. Training, and a sudden jump
4. The two circuits of a head
5. Reading OV: who copies?
6. Reading QK: where does each head look?
7. Composition: how layer 0 helps layer 1
8. Checking with a forward pass and ablations
9. Back to chapter 2: what Q, K, V learn
10. Summary, questions, exercises

In [ ]:
# On Google Colab: fetch the course repo (wsp.py lives next to this notebook).
# Locally, this cell does nothing.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("wsp.py"):
    if not os.path.exists("ai-masterclass"):
        !git clone -q --depth 1 https://github.com/zenocode-org/ai-masterclass
    %cd ai-masterclass/en/references/01_weight_space_programming

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

import wsp
torch.manual_seed(0)
torch.set_grad_enabled(True)
VOCAB, D_MODEL, N_HEAD, N_LAYER, CTX = 48, 64, 4, 2, 48

## 1. The task: copy a repeated segment

Each sequence is 48 random tokens (from a vocabulary of 48). Somewhere inside it, a random segment of 6 to 16 tokens is **repeated** right after itself, at a random position.

Random tokens cannot be predicted (best loss = ln 48 ≈ 3.87). But once the repetition has started, the next token **can** be predicted by looking back: *"the current token appeared earlier; whatever followed it then will follow it now."* That rule is called **induction**. It is the simplest form of in-context learning, and models do it by copying.

We measure the loss **only on the predictable positions** (the `mask`).

In [ ]:
x, y, mask, src = wsp.repeated_batch(1, VOCAB, CTX, torch.Generator().manual_seed(1))
print("tokens     :", x[0].tolist())
print("predictable:", "".join("^" if m else "." for m in mask[0].tolist()))
rep = torch.nonzero(src[0] >= 0).flatten()
print(f"positions {rep[0].item()}..{rep[-1].item()} repeat positions {src[0, rep[0]].item()}..{src[0, rep[-1]].item()}")

## 2. A transformer made only of attention heads

To make every computation readable, we strip the chapter 2 mini-GPT down to the bone:

- token embedding `E` + **learned absolute position embedding** `P` (as in chapter 2);
- **2 attention layers × 4 heads**, residual connections;
- **no MLP and no LayerNorm**, so the only things that move information are heads;
- unembedding `U` → logits.

`wsp.AttnOnly` is ~40 lines. Each layer stores Q, K, V, O as four `d×d` matrices, and `head_weights(h)` slices out head `h`'s part (`hs = 64/4 = 16` dimensions).

In [ ]:
model = wsp.AttnOnly(VOCAB, D_MODEL, N_HEAD, N_LAYER, CTX)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

## 3. Training, and a sudden jump

In [ ]:
torch.manual_seed(0)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
xv, yv, mv, sv = wsp.repeated_batch(512, VOCAB, CTX, torch.Generator().manual_seed(123))  # fixed eval set
curve = []
for it in range(6000):
    xb, yb = wsp.repeated_batch(64, VOCAB, CTX)[:2]
    logits, _ = model(xb)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), yb.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if it % 100 == 0:
        curve.append((it, wsp.repeated_loss(model, xv, yv, mv)))
torch.set_grad_enabled(False)

plt.plot(*zip(*curve)); plt.axhline(3.87, ls=":", c="gray", label="ln 48 (guessing)")
plt.xlabel("step"); plt.ylabel("loss on predictable positions"); plt.legend(); plt.title("Learning to copy"); plt.show()
print(f"final loss on predictable positions: {curve[-1][1]:.2f}")

The loss stays flat near "guessing" for a long time, then **drops suddenly**. Olsson et al. (2022) saw the same jump in real language models and tied it to the formation of **induction heads**. A circuit that needs two heads to cooperate is useless until both pieces exist, so it appears all at once.

Now the real question: **which heads do what?** We answer it first **from weights only**, like Astra in Experiment 3, and only check with a forward pass at the end.

## 4. The two circuits of a head

For a query position $i$ and a key position $j$, a head computes

$$\text{score}(i,j) = (W_Q x_i)\cdot(W_K x_j) = x_i^\top \underbrace{W_Q^\top W_K}_{W_{QK}}\, x_j \qquad\qquad \text{out}_i = \sum_j \text{softmax}(\text{score})_{ij}\; \underbrace{W_O W_V}_{W_{OV}}\, x_j$$

So a head is **two independent programs**:

| Circuit | Matrix | Decides |
|---|---|---|
| **QK** | $W_{QK}=W_Q^\top W_K$ ($d\times d$) | *where* to look: which key positions get attention |
| **OV** | $W_{OV}=W_O W_V$ ($d\times d$) | *what* to move: what gets written into the residual stream once attention is fixed |

`W_Q` and `W_K` are never meaningful **alone**. You can multiply `W_Q` by any invertible matrix `A` and `W_K` by `A⁻ᵀ` without changing the model. Only the product is real. Same for `W_V` and `W_O`.

To read a $d\times d$ matrix we **sandwich it** between things we understand: the token embeddings `E` (48×64), the position embeddings `P` (48×64), and the unembedding `U` (48×64).

In [ ]:
E, P, U = model.E.weight, model.P.weight, model.U.weight   # (48, 64) each

def circuits(layer, head):
    WQ, WK, WV, WO = model.layers[layer].head_weights(head)
    return WQ.T @ WK, WO @ WV          # W_QK (64x64), W_OV (64x64)

WQK, WOV = circuits(0, 0)
print("W_QK:", tuple(WQK.shape), "| W_OV:", tuple(WOV.shape))
print("rank of W_QK:", torch.linalg.matrix_rank(WQK).item(), "| rank of W_OV:", torch.linalg.matrix_rank(WOV).item(),
      "(at most head_size = 16: a head reads and writes a 16-dim slice)")

## 5. Reading OV: who copies?

$U\,W_{OV}\,E^\top$ is a 48×48 table. Entry $(s,t)$ = *"if this head attends to token $t$, how much does it raise the logit of token $s$?"* A **copying head** has a bright diagonal: attending to token $t$ raises $t$.

A one-number summary (Elhage et al.) is the **copying score** $\sum_k \lambda_k / \sum_k |\lambda_k|$ over the eigenvalues of that table. It is +1 for pure copying, −1 for pure anti-copying, and near 0 for an unrelated mapping.

In [ ]:
def ov_table(layer, head):
    return U @ circuits(layer, head)[1] @ E.T                 # (vocab, vocab)

def copying_score(M):
    ev = torch.linalg.eigvals(M)
    return (ev.sum().real / ev.abs().sum()).item()

fig, axes = plt.subplots(N_LAYER, N_HEAD, figsize=(11, 5.5))
for l in range(N_LAYER):
    for h in range(N_HEAD):
        M = ov_table(l, h)
        axes[l, h].imshow(M, cmap="RdBu", vmin=-M.abs().max(), vmax=M.abs().max())
        axes[l, h].set_title(f"L{l}H{h}  copy={copying_score(M):+.2f}", fontsize=9)
        axes[l, h].set_xticks([]); axes[l, h].set_yticks([])
fig.suptitle("OV circuit  U·W_OV·Eᵀ   (column = token attended to, row = logit raised)")
plt.tight_layout(); plt.show()

All four **layer-1** heads have a clean diagonal and a copying score near +1: **they copy whatever token they attend to** straight into the logits. They are the copying heads Experiment 3 asked about.

The layer-0 OV tables look weaker. Careful: that does *not* mean layer 0 does nothing. `U W_OV Eᵀ` only measures effects **on the output logits**. A layer-0 head may write something meant for **layer 1** to read, not for the output. We come back to this in section 7.

## 6. Reading QK: where does each head look?

Attention scores mix two kinds of information in $x_j = E[\text{token}_j] + P[j]$. We read them separately:

- **token → token**: $E\,W_{QK}\,E^\top$. A bright diagonal means "a query attends to keys with **the same token**".
- **position → position**: $P\,W_{QK}\,P^\top$, with the causal mask. A bright line just below the diagonal would mean "attend to the **previous** position". That is the textbook *previous-token head*.

In [ ]:
causal = torch.triu(torch.ones(CTX, CTX, dtype=torch.bool), 1)

def qk_token(layer, head):
    return E @ circuits(layer, head)[0] @ E.T

def qk_pos(layer, head):
    return (P @ circuits(layer, head)[0] @ P.T).masked_fill(causal, float("nan"))

fig, axes = plt.subplots(2, N_HEAD, figsize=(11, 5.5))
for h in range(N_HEAD):
    T_ = qk_token(0, h); Pm = qk_pos(0, h)
    dup = (T_.argmax(1) == torch.arange(VOCAB)).float().mean().item()
    axes[0, h].imshow(T_, cmap="RdBu", vmin=-T_.abs().max(), vmax=T_.abs().max())
    axes[0, h].set_title(f"L0H{h} token→token\nsame-token argmax: {dup:.0%}", fontsize=9)
    axes[1, h].imshow(Pm, cmap="RdBu"); axes[1, h].set_title(f"L0H{h} position→position", fontsize=9)
    for a in axes[:, h]: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

Read the plots:

- **Token → token:** every layer-0 head has a strong **diagonal**. Each one is a **duplicate-token head**: a query token looks for *earlier occurrences of itself*.
- **Position → position:** there is **no** bright line below the diagonal. This model did **not** learn a previous-token head.

This is a good reading lesson. The classic induction circuit (Olsson et al.) is *previous-token head in layer 0 → induction head in layer 1*. Our model found **a different algorithm** that works just as well. If we had assumed the textbook circuit, we would have misread it. **Read what is there, not what you expect.** (With another seed, you may get the textbook version. Exercise 1 asks you to check.)

So layer 0 finds *"where did my token appear before?"*. How does layer 1 then know *what came next*?

## 7. Composition: how layer 0 helps layer 1

Hypothesis: when a layer-0 head attends to the earlier copy at position $j$, its OV circuit **carries the position embedding** $P[j]$ into the current position. A layer-1 head can then use that as its **query**: *"find the key at position $j+1$"*, the token right after the earlier copy. The OV of that layer-1 head copies it to the output (section 5).

That is **Q-composition**: the output of a layer-0 head feeds the **query** side of a layer-1 head. We can test it with weights only. The query built from a moved position is $W_{OV}^{(0)} P[j]$ and the key at position $k$ contains $P[k]$:

$$S_{jk} = \big(W_{OV}^{(0)}P[j]\big)^\top W_{QK}^{(1)}\, P[k]$$

If the hypothesis is right, row $j$ of $S$ peaks at $k=j+1$.

In [ ]:
def q_composition(h0, h1):
    WOV0 = circuits(0, h0)[1]
    WQK1 = circuits(1, h1)[0]
    return (P @ WOV0.T) @ WQK1 @ P.T                     # S[j, k]

def plus_one_rate(S):
    return (S.argmax(1)[:-1] == torch.arange(1, CTX)).float().mean().item()

table = torch.tensor([[plus_one_rate(q_composition(h0, h1)) for h1 in range(N_HEAD)] for h0 in range(N_HEAD)])
print("share of positions j where layer-1 head attends to j+1, via each layer-0 head's OV:")
print("        " + "  ".join(f"L1H{h}" for h in range(N_HEAD)))
for h0 in range(N_HEAD):
    print(f"L0H{h0}   " + "  ".join(f"{v:4.0%}" for v in table[h0]))

h0_star = int(table.mean(1).argmax())
S = q_composition(h0_star, 0)
plt.imshow(S, cmap="RdBu", vmin=-S.abs().max(), vmax=S.abs().max())
plt.xlabel("key position k"); plt.ylabel("position j carried by L0's OV")
plt.title(f"L0H{h0_star} → L1H0: the bright band is k = j+1"); plt.show()

One layer-0 head stands out: through its OV circuit, **every** layer-1 head's QK circuit points to $j+1$. The whole algorithm has now been read from weights:

1. **Layer 0, the duplicate-token head (QK):** "find the earlier occurrence of my token".
2. **Layer 0, the same head (OV):** "bring back its **position** $j$".
3. **Layer 1 (QK, Q-composition):** "attend to position $j+1$".
4. **Layer 1 (OV, copying):** "copy that token to the output".

Experiment 3 did this on a 4-layer, 6-head model of real text, with no forward pass, and its copying-head list matched the measured ground truth.

## 8. Checking with a forward pass and ablations

Astra was not allowed to do this in Experiment 3. We are. Two checks:

- **Attention patterns:** layer-1 heads should attend to `src + 1` (the token after the earlier copy). The duplicate-token head should attend to `src`.
- **Ablation:** zero one head's output and measure how much the loss on predictable positions rises. A head on the critical path should hurt a lot.

In [ ]:
logits, patterns = model(xv)
bi, ti = torch.nonzero(sv >= 0, as_tuple=True)
base = wsp.repeated_loss(model, xv, yv, mv)
print(f"loss on predictable positions: {base:.3f}\n")
print("head    attn→src   attn→src+1   loss increase if ablated")
for l in range(N_LAYER):
    for h in range(N_HEAD):
        pat = patterns[l][:, h]
        to_src = pat[bi, ti, sv[bi, ti]].mean().item()
        to_next = pat[bi, ti, (sv[bi, ti] + 1).clamp(max=CTX - 1)].mean().item()
        delta = wsp.repeated_loss(model, xv, yv, mv, ablate=(l, h)) - base
        print(f"L{l}H{h}    {to_src:7.2f}   {to_next:10.2f}   {delta:+8.3f}")

In [ ]:
b = 0
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(patterns[0][b, h0_star], cmap="Blues"); axes[0].set_title(f"L0H{h0_star}: attends to the earlier copy (src)")
axes[1].imshow(patterns[1][b, 0], cmap="Blues"); axes[1].set_title("L1H0: attends one step after it (src+1)")
for a in axes: a.set_xlabel("key position"); a.set_ylabel("query position")
plt.tight_layout(); plt.show()

The forward pass confirms the weight reading: the duplicate-token head looks at `src`, and the layer-1 heads look at `src + 1`. Ablating the duplicate-token head breaks the copying almost entirely. The four layer-1 heads share the work, so removing one hurts, but less.

> **Link to Experiment 7.** When Astra had to find a sycophantic Gemma-3 from **weights alone**, it failed 4 times out of 4. With **attention patterns** on texts it wrote, it succeeded 3 out of 3, through a head (10.2) that looks at the user's suggested answer instead of the question. Reading weights is powerful on small, clean models like this one. On a real LLM, forward-pass evidence is often what makes the reading reliable.

## 9. Back to chapter 2: what do Q, K and V learn?

In chapter 2 we said: *query = "what I look for", key = "what I contain", value = "what I transmit"*. This notebook makes those phrases concrete and testable:

- **Q and K learn a *matching rule*, and only as a pair.** Here, layer 0's $W_{QK}$ learned *"same token"* (a diagonal in token space), and layer 1's learned *"position + 1"* (an off-diagonal in position space). The rule is visible once you sandwich $W_{QK}$ between the embeddings.
- **V and O learn a *message*, also only as a pair.** Layer 1's $W_{OV}$ learned *"copy the token"*. Layer 0's learned *"send my position"*, a message meant for another head, not for the output.
- **Heads compose.** A head's QK can use what an earlier head's OV wrote. That is how two weak operations become an algorithm.

---
## 10. Summary

- A head = **QK circuit** (where to look) + **OV circuit** (what to move). Only the products $W_Q^\top W_K$ and $W_O W_V$ matter.
- Sandwich them with `E`, `P`, `U` to read them as tables over tokens and positions. The **copying score** summarizes OV.
- Heads in different layers **compose** (Q-, K- or V-composition), and that composition can also be read from weights.
- Trained models may find an **unexpected algorithm**. Read what is there.

### Check your understanding
1. Why can't you interpret `W_Q` alone? Give the transformation that changes `W_Q` but not the model.
2. A head has copying score −0.9. What does it do to the logits of the tokens it attends to?
3. Why does the loss stay flat for a long time and then drop suddenly?
4. In section 5, the layer-0 OV tables looked weak. Why was it wrong to conclude that layer 0 is useless?

### Exercises
1. **Another seed.** Retrain with `torch.manual_seed(1)` and `2`. Do you always get duplicate-token heads, or sometimes a previous-token head (a line below the diagonal in `P W_QK Pᵀ`)? Write a function that classifies each layer-0 head automatically.
2. **Force the textbook circuit.** Remove the position embedding from the **value** path only (feed `E[token]` to `W_V` but `E + P` to `W_Q`, `W_K`). The duplicate-token trick can no longer carry positions. What does the model learn instead?
3. **K-composition.** Write the K-composition score $\lVert W_{QK}^{(1)}\,W_{OV}^{(0)}\rVert_F / (\lVert W_{QK}^{(1)}\rVert_F\,\lVert W_{OV}^{(0)}\rVert_F)$ for every pair. Does it find the circuit here? Why or why not?
4. **Chapter 2 model.** Load the mini-GPT trained in `chapter_2/09` and compute the copying score of each head (`U W_O W_V Eᵀ` needs care: there is a LayerNorm before `lm_head` and an MLP between blocks). Which heads copy?

### References
- N. Elhage et al., *A Mathematical Framework for Transformer Circuits* (2021): <https://transformer-circuits.pub/2021/framework/index.html>
- C. Olsson et al., *In-context Learning and Induction Heads* (2022): <https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html>
- Experiment 3 logs: `exp3_language/` in <https://github.com/RikiyaT/lm_understand-logs> (Astra's Python code for finding copying heads is in `runs/*/tool_calls.jsonl`).